In [1]:
# Cell 1: Mount Drive and set workspace
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline


In [2]:
# Cell 2: Install API dependencies
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "-q"
])

pkgs = [
    "openai>=1.55.0",
    "tqdm==4.66.2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import openai
import tqdm

print("openai:", openai.__version__)
print("tqdm:", tqdm.__version__)
print("Installed OK")

openai: 2.32.0
tqdm: 4.66.2
Installed OK


In [3]:
# Cell 3: Clone or update repository
import os
import subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Pulling latest changes...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

print("Repo ready at:", REPO_DIR)
print("Repo root files:", os.listdir(REPO_DIR)[:15])

Cloning repository into: /content/KGP-CuriousLLM
Repo ready at: /content/KGP-CuriousLLM
Repo root files: ['create_dirs.py', 'ft_mistral_main.py', 'KGP', 'configs', '.gitignore', 'MDR_main.py', 'kgp_main.py', 'requirements.txt', 'images', 'kg_construct_main.py', 'MDR_embedding_main.py', 'quantize_mistral_main.py', '.git', 'T5_main.py', 'grid_search_mistral_main.py']


In [4]:
# Cell 4: Add repo and baseline to Python path
import os
import sys

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("PYTHONPATH ready")
print("BASE_DIR in path:", BASE_DIR in sys.path)
print("REPO_DIR in path:", REPO_DIR in sys.path)

PYTHONPATH ready
BASE_DIR in path: True
REPO_DIR in path: True


In [5]:
# Cell 5: Set HotpotQA paths and Qwen config
import os

MODEL_NAME = "Qwen/Qwen3.5-9B"

DATA_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "hotpot_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "answers",
    "hotpot_answers_qwen3_5_9b",
)

SAVE_PATH = os.path.join(
    SAVE_DIR,
    "qwen_agent_responses.json",
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("MODEL_NAME =", MODEL_NAME)
print("DATA_PATH  =", DATA_PATH)
print("SAVE_DIR   =", SAVE_DIR)
print("SAVE_PATH  =", SAVE_PATH)

assert os.path.isfile(DATA_PATH), f"Missing evidence file: {DATA_PATH}"
print("Evidence file exists.")

MODEL_NAME = Qwen/Qwen3.5-9B
DATA_PATH  = /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/evidence.json
SAVE_DIR   = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_qwen3_5_9b
SAVE_PATH  = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_qwen3_5_9b/qwen_agent_responses.json
Evidence file exists.


In [6]:
# Cell 6A: Install Hugging Face deps without changing torch
import sys
import subprocess
import os

# Remove packages that caused CUDA/import conflicts.
subprocess.call(
    [sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes", "torchvision", "torchaudio"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Install only text-generation dependencies.
pkgs = [
    "transformers>=5.0.0",
    "accelerate",
    "sentencepiece",
    "protobuf",
    "huggingface-hub",
    "safetensors",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs)

# Avoid stale failed imports in the current kernel.
for m in list(sys.modules.keys()):
    if m.startswith("torchvision") or m.startswith("bitsandbytes"):
        del sys.modules[m]

import torch
import transformers
import accelerate
import huggingface_hub

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("HF dependencies ready.")

torch: 2.10.0+cu128
transformers: 5.8.0
accelerate: 1.13.0
huggingface_hub: 1.14.0
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory GB: 39.49
HF dependencies ready.


In [7]:
# Cell 6: Load Qwen3.5-9B from Hugging Face in BF16
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3.5-9B"

assert torch.cuda.is_available(), "GPU is required for local Qwen3.5-9B inference."

device = torch.device("cuda:0")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model = model.to(device)
model.eval()

print("Loaded model:", MODEL_NAME)
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded model: Qwen/Qwen3.5-9B
Model device: cuda:0
Model dtype: torch.bfloat16
pad_token: <|endoftext|>
eos_token: <|im_end|>


In [8]:
# Cell 7: Load and inspect evidence file
import json
from collections import Counter

def load_json(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

data = load_json(DATA_PATH)

print("num_records =", len(data))
print("type_counts =", Counter(r.get("type", "unknown") for r in data))
print("first_keys =", list(data[0].keys()))

print("\nFirst question:")
print(data[0]["question"])

print("\nFirst answer:")
print(data[0].get("answer", ""))

print("\nFirst evidence preview:")
for i, ev in enumerate(data[0].get("evidence", [])[:3], start=1):
    print(f"{i}.", ev[:500])

num_records = 1000
type_counts = Counter({'bridge': 700, 'comparison': 300})
first_keys = ['type', 'question', 'evidence', 'answer', 'supports']

First question:
Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?

First answer:
yes

First evidence preview:
1. Title: Operation Dragoon. Evidence: Operation Dragoon also had political implications.
2. Title: Operation Dragoon. Evidence: Despite these successes, there was criticism of Dragoon by some Allied generals and contemporary commentators such as Bernard Montgomery, Arthur R. Wilson, and Chester Wilmot in the aftermath, mostly because of its geo-strategic implications.
3. Title: Operation Dragoon. Evidence: In the northeast the German problems loomed as large.


In [9]:
# Cell 8: Define repo-faithful prompts

prompt = """
    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
    - Do not include any additional tokens, explanations, or information beyond the direct answer.

    QUESTION: {question}
    CONTEXT: {context}
    ANSWER: [Your concise answer here or "Information not available" if the answer cannot be determined from the contexts.]

    """

none_prompt = """Given the following question, create a final answer in English to the question.
    QUESTION: {question}
    ANSWER: [Please provide only the answer and keep the answer less than 6 words.]
    """

print("Prompts ready.")

Prompts ready.


In [10]:
# Cell 9: Define response cleaner
import re

def clean_qwen_response(text: str) -> str:
    # Remove technical or chat artifacts if the backend returns them.
    if text is None:
        return ""

    text = text.strip()

    # Remove possible thinking block if backend ignores non-thinking flag.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # Remove common prefixes.
    for prefix in ["ANSWER:", "Answer:", "answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    # Keep first line if model adds extra explanation.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0].strip()

    # Remove quotes around short answers.
    text = text.strip().strip('"').strip("'").strip()

    return text

print("Cleaner ready.")

Cleaner ready.


In [11]:
# Cell 10: Define local Qwen non-thinking generation
import torch

def build_qwen_prompt(input_prompt: str):
    messages = [
        {"role": "user", "content": input_prompt},
    ]

    # Qwen non-thinking mode through chat template if supported.
    try:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return prompt_text


@torch.inference_mode()
def qwen_chat_completion(input_prompt: str, max_tokens: int = 30):
    prompt_text = build_qwen_prompt(input_prompt)

    model_device = next(model.parameters()).device

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    )

    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
    resp = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return clean_qwen_response(resp)

print("Local Qwen non-thinking generation function ready.")

Local Qwen non-thinking generation function ready.


In [12]:
# Cell 11: Build one input prompt for smoke test
sample = data[0]

sample_type = sample["type"]
sample_question = sample["question"]
sample_contexts = sample["evidence"]
sample_gt = sample["answer"]

if sample_contexts:
    sample_contexts_text = "\n".join(
        f"{i}: {c}" for i, c in enumerate(sample_contexts, start=1)
    )
    sample_input_prompt = prompt.format(
        question=sample_question,
        context=sample_contexts_text,
    )
else:
    sample_input_prompt = none_prompt.format(question=sample_question)

print("Sample type:", sample_type)
print("Sample question:", sample_question)
print("Sample gt:", sample_gt)
print("\nPrompt preview:")
print(sample_input_prompt[:3000])

Sample type: comparison
Sample question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
Sample gt: yes

Prompt preview:

    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
    - Do not include any additional tokens, explanations, or information beyond the direct answer.

    QUESTION: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
    CONTEXT: 1: Title: Operation Dragoon. Evidence: Operation Dragoon also had political implication

In [13]:
# Cell 12: Smoke test local Qwen on one sample
sample_response = qwen_chat_completion(
    input_prompt=sample_input_prompt,
    max_tokens=30,
)

print("Question:", sample_question)
print("GT:", sample_gt)
print("Qwen response:", sample_response)

Question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
GT: yes
Qwen response: World War II and the American Civil War.


In [14]:
# Cell 13: Define repo-style answer generation pipeline
import os
import json
from tqdm import tqdm

def pipeline(data, save_path):
    responses = []

    for record in tqdm(data, total=len(data)):
        q_type = record["type"]
        question = record["question"]
        contexts = record["evidence"]
        gt = record["answer"]

        if contexts:
            contexts = "\n".join(
                f"{i}: {c}" for i, c in enumerate(contexts, start=1)
            )
            input_prompt = prompt.format(
                question=question,
                context=contexts,
            )
        else:
            input_prompt = none_prompt.format(question=question)

        resp = qwen_chat_completion(
            input_prompt=input_prompt,
            max_tokens=30,
        )

        response = {
            "type": q_type,
            "question": question,
            "gt": gt,
            "response": resp,
        }

        responses.append(response)

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(responses, f, indent=4, ensure_ascii=False)

    return responses

print("Pipeline ready.")

Pipeline ready.


In [15]:
# Cell 14: Run answer generation for HotpotQA
responses = pipeline(
    data=data,
    save_path=SAVE_PATH,
)

print("Finished.")
print("Saved to:", SAVE_PATH)
print("Total responses:", len(responses))

100%|██████████| 1000/1000 [13:30<00:00,  1.23it/s]

Finished.
Saved to: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_qwen3_5_9b/qwen_agent_responses.json
Total responses: 1000


In [16]:
# Cell 15: Verify saved answers
import os
import json
from collections import Counter

assert os.path.isfile(SAVE_PATH), f"Missing output file: {SAVE_PATH}"

saved = load_json(SAVE_PATH)

print("SAVE_PATH =", SAVE_PATH)
print("num_saved =", len(saved))
print("type_counts =", Counter(r.get("type", "unknown") for r in saved))

print("\nFirst saved response:")
print(json.dumps(saved[0], indent=2, ensure_ascii=False)[:3000])

SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_qwen3_5_9b/qwen_agent_responses.json
num_saved = 1000
type_counts = Counter({'bridge': 700, 'comparison': 300})

First saved response:
{
  "type": "comparison",
  "question": "Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?",
  "gt": "yes",
  "response": "World War II and the American Civil War."
}


In [17]:
# Cell 16: Set 2WikiMQA paths

import os

TWO_WIKI_DATA_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "2wikimultihopqa_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

TWO_WIKI_SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "answers",
    "wiki_answers_qwen3_5_9b",
)

TWO_WIKI_SAVE_PATH = os.path.join(
    TWO_WIKI_SAVE_DIR,
    "qwen_agent_responses.json",
)

os.makedirs(TWO_WIKI_SAVE_DIR, exist_ok=True)

print("TWO_WIKI_DATA_PATH =", TWO_WIKI_DATA_PATH)
print("TWO_WIKI_SAVE_DIR  =", TWO_WIKI_SAVE_DIR)
print("TWO_WIKI_SAVE_PATH =", TWO_WIKI_SAVE_PATH)

assert os.path.isfile(TWO_WIKI_DATA_PATH), f"Missing 2Wiki evidence file: {TWO_WIKI_DATA_PATH}"
print("2Wiki evidence file exists.")

TWO_WIKI_DATA_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/2wikimultihopqa_evidence_1000/qwen_agent/evidence.json
TWO_WIKI_SAVE_DIR  = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_qwen3_5_9b
TWO_WIKI_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_qwen3_5_9b/qwen_agent_responses.json
2Wiki evidence file exists.


In [18]:
# Cell 17: Load and inspect 2WikiMQA evidence

import json
from collections import Counter

two_wiki_data = load_json(TWO_WIKI_DATA_PATH)

print("num_records =", len(two_wiki_data))
print("type_counts =", Counter(r.get("type", "unknown") for r in two_wiki_data))
print("first_keys =", list(two_wiki_data[0].keys()))

print("\nFirst question:")
print(two_wiki_data[0]["question"])

print("\nFirst answer:")
print(two_wiki_data[0].get("answer", ""))

print("\nFirst evidence preview:")
for i, ev in enumerate(two_wiki_data[0].get("evidence", [])[:5], start=1):
    print(f"{i}.", ev[:500])

num_records = 1000
type_counts = Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})
first_keys = ['type', 'question', 'evidence', 'answer', 'supports']

First question:
Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?

First answer:
yes

First evidence preview:
1. Title: Pauline Auzou. Evidence: Taylor & Francis; January 1997. . p. 199.
2. Title: Martial Law (1991 film). Evidence: The film has yet to arrive onto DVD in the United States.
3. Title: Pauline Auzou. Evidence: Berg; 6 April 1995. . p. 34.
4. Title: Jean Rollin. Evidence: Le temps d'un visage (1990), Jean Rollin. Éd.
5. Title: Até que a Sorte nos Separe. Evidence: Até que a Sorte nos Separe (English: Till Luck Do Us Part) is a 2012 Brazilian comedy film directed by Roberto Santucci and starring Leandro Hassum and Danielle Winits.


In [19]:
# Cell 18: Smoke test Qwen on one 2WikiMQA sample

two_wiki_sample = two_wiki_data[0]

two_wiki_sample_type = two_wiki_sample["type"]
two_wiki_sample_question = two_wiki_sample["question"]
two_wiki_sample_contexts = two_wiki_sample["evidence"]
two_wiki_sample_gt = two_wiki_sample["answer"]

if two_wiki_sample_contexts:
    two_wiki_sample_contexts_text = "\n".join(
        f"{i}: {c}" for i, c in enumerate(two_wiki_sample_contexts, start=1)
    )
    two_wiki_sample_input_prompt = prompt.format(
        question=two_wiki_sample_question,
        context=two_wiki_sample_contexts_text,
    )
else:
    two_wiki_sample_input_prompt = none_prompt.format(
        question=two_wiki_sample_question
    )

two_wiki_sample_response = qwen_chat_completion(
    input_prompt=two_wiki_sample_input_prompt,
    max_tokens=30,
)

print("Sample type:", two_wiki_sample_type)
print("Question:", two_wiki_sample_question)
print("GT:", two_wiki_sample_gt)
print("Qwen response:", two_wiki_sample_response)

Sample type: bridge_comparison
Question: Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?
GT: yes
Qwen response: No, they are from different countries.


In [20]:
# Cell 19: Define resume-safe 2WikiMQA answer generation pipeline

import os
import json
from tqdm import tqdm

def pipeline_resume(data, save_path):
    if os.path.isfile(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            responses = json.load(f)

        done_questions = {r["question"] for r in responses}
        print("Resuming from existing output.")
        print("Existing responses:", len(responses))
    else:
        responses = []
        done_questions = set()
        print("No existing output found. Starting from scratch.")

    for record in tqdm(data, total=len(data)):
        q_type = record["type"]
        question = record["question"]
        contexts = record["evidence"]
        gt = record["answer"]

        if question in done_questions:
            continue

        if contexts:
            contexts = "\n".join(
                f"{i}: {c}" for i, c in enumerate(contexts, start=1)
            )
            input_prompt = prompt.format(
                question=question,
                context=contexts,
            )
        else:
            input_prompt = none_prompt.format(question=question)

        resp = qwen_chat_completion(
            input_prompt=input_prompt,
            max_tokens=30,
        )

        response = {
            "type": q_type,
            "question": question,
            "gt": gt,
            "response": resp,
        }

        responses.append(response)
        done_questions.add(question)

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(responses, f, indent=4, ensure_ascii=False)

    return responses

print("Resume-safe pipeline ready.")

Resume-safe pipeline ready.


In [21]:
# Cell 20: Run answer generation for 2WikiMQA

two_wiki_responses = pipeline_resume(
    data=two_wiki_data,
    save_path=TWO_WIKI_SAVE_PATH,
)

print("Finished.")
print("Saved to:", TWO_WIKI_SAVE_PATH)
print("Total responses:", len(two_wiki_responses))

No existing output found. Starting from scratch.


100%|██████████| 1000/1000 [14:19<00:00,  1.16it/s]

Finished.
Saved to: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_qwen3_5_9b/qwen_agent_responses.json
Total responses: 1000


In [22]:
# Cell 21: Verify saved 2WikiMQA answers

import os
import json
from collections import Counter

assert os.path.isfile(TWO_WIKI_SAVE_PATH), f"Missing output file: {TWO_WIKI_SAVE_PATH}"

two_wiki_saved = load_json(TWO_WIKI_SAVE_PATH)

print("TWO_WIKI_SAVE_PATH =", TWO_WIKI_SAVE_PATH)
print("num_saved =", len(two_wiki_saved))
print("type_counts =", Counter(r.get("type", "unknown") for r in two_wiki_saved))

print("\nFirst saved response:")
print(json.dumps(two_wiki_saved[0], indent=2, ensure_ascii=False)[:3000])

TWO_WIKI_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_qwen3_5_9b/qwen_agent_responses.json
num_saved = 1000
type_counts = Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})

First saved response:
{
  "type": "bridge_comparison",
  "question": "Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?",
  "gt": "yes",
  "response": "No, the directors are from different countries."
}


In [23]:
# Cell 22: Check output schema consistency

required_keys = {"type", "question", "gt", "response"}

bad_records = []
for i, record in enumerate(two_wiki_saved):
    if set(record.keys()) != required_keys:
        bad_records.append((i, list(record.keys())))

print("Expected keys:", required_keys)
print("Bad records:", len(bad_records))

if bad_records:
    print("First bad record:", bad_records[0])
else:
    print("All records have the expected repo-style schema.")

Expected keys: {'response', 'type', 'question', 'gt'}
Bad records: 0
All records have the expected repo-style schema.


In [24]:
# Cell 25: Final path summary

print("HotpotQA answers:")
print(SAVE_PATH)

print("\n2WikiMQA answers:")
print(TWO_WIKI_SAVE_PATH)

print("\nFiles exist:")
print("HotpotQA:", os.path.isfile(SAVE_PATH))
print("2WikiMQA:", os.path.isfile(TWO_WIKI_SAVE_PATH))

HotpotQA answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_qwen3_5_9b/qwen_agent_responses.json

2WikiMQA answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_qwen3_5_9b/qwen_agent_responses.json

Files exist:
HotpotQA: True
2WikiMQA: True
